# 02 · Prepare Dataset
Build a career-coaching instruction dataset and save it as `data/train.jsonl` and `data/eval.jsonl`.

## 2.1 · Seed examples
> Add as many examples as you can here. Aim for **2,000+** pairs before serious training.

In [1]:
from datasets import load_dataset

hf_data = load_dataset("vicgalle/alpaca-gpt4", split="train")

# Filter for career-related rows
career_keywords = ["resume", "interview", "job", "career", "salary", "cover letter", "linkedin"]
career_rows = [
    {"instruction": r["instruction"], "input": r["input"], "output": r["output"]}
    for r in hf_data
    if any(kw in (r["instruction"] + r["input"]).lower() for kw in career_keywords)
]
print(f"Found {len(career_rows)} career-related examples")
# Merge with SEED_EXAMPLES before calling save_jsonl()

Found 748 career-related examples


In [2]:
SEED_EXAMPLES = [
    {
        "instruction": "Review this resume summary and rewrite it to be stronger.",
        "input": "Hardworking software developer with experience in many technologies. Looking for a challenging role where I can grow.",
        "output": "Full-stack engineer with 3 years of production experience building scalable Node.js APIs and React frontends serving 50K+ users. Seeking a senior engineering role where I can drive architecture decisions and mentor junior developers.",
    },
    {
        "instruction": "What keywords is this resume missing for a backend engineering role?",
        "input": "Skills: Python, Django, MySQL, HTML, CSS. Experience: Built internal tools at a fintech startup.",
        "output": "Your resume is missing several high-signal backend keywords:\n• REST / GraphQL API design\n• Docker and Kubernetes\n• Redis or Celery (async queues)\n• CI/CD (GitHub Actions, Jenkins)\n• Cloud platform (AWS, GCP, or Azure)\n• PostgreSQL\n• System design / microservices\n\nAdd any you genuinely have experience with, and quantify impact.",
    },
    {
        "instruction": "How should I format my resume for ATS systems?",
        "input": "",
        "output": "To pass ATS:\n1. Single-column layout — no tables or text boxes.\n2. Save as .docx or plain PDF.\n3. Standard headings: Experience, Education, Skills.\n4. Mirror keywords from the job description verbatim.\n5. Avoid headers/footers for contact info.\n6. Standard fonts (Arial, Calibri) at 10–12pt.\n7. Spell out acronyms once: 'Natural Language Processing (NLP)'.",
    },
    {
        "instruction": "Rewrite this bullet point to show measurable impact.",
        "input": "Responsible for managing the database and making it faster.",
        "output": "Optimised PostgreSQL query performance by introducing composite indexes and caching, reducing average response time from 1.2s to 180ms (85% improvement) for the 3 most-used API endpoints.",
    },
    {
        "instruction": "What is the STAR method and how do I use it for interview answers?",
        "input": "",
        "output": "STAR = Situation, Task, Action, Result.\nS: Set the scene briefly.\nT: What was your specific responsibility?\nA: What did YOU do? Use 'I', be specific.\nR: Quantify the outcome (%, time saved, revenue).\n\nExample:\nS: Our deployment pipeline took 45 minutes with manual steps.\nT: I was asked to reduce deployment time.\nA: I automated it with GitHub Actions and added rollback triggers.\nR: Deployment time dropped to 8 minutes (82% reduction).",
    },
    {
        "instruction": "How do I write a cover letter for a software engineering role?",
        "input": "I am applying for a Senior Backend Engineer position at a fintech startup.",
        "output": "Keep it to 3 short paragraphs:\n\nPara 1 — The hook: Open with what excites you about THIS company specifically.\nPara 2 — The evidence: Your strongest relevant win, connected to their needs.\nPara 3 — The close: 'I'd welcome the chance to discuss how I can help. Available any time this week.'\n\nKeep it under 250 words.",
    },
    {
        "instruction": "What are common resume mistakes to avoid?",
        "input": "",
        "output": "8 common mistakes:\n1. Responsibilities instead of achievements.\n2. Missing numbers — every bullet needs 'how much?'\n3. One resume for all jobs — tailor keywords.\n4. Generic objective statement.\n5. Irrelevant experience older than 10 years.\n6. Spelling and grammar errors.\n7. Fancy formatting — tables and icons confuse ATS.\n8. Too long — 1 page under 5 years, 2 pages maximum.",
    },
    {
        "instruction": "How do I negotiate a salary offer?",
        "input": "I received an offer of ₹12 LPA for a mid-level software engineer role in Bangalore.",
        "output": "Steps:\n1. Research: Glassdoor/Levels.fyi show ₹15–22 LPA is typical for mid-level SWE in Bangalore.\n2. Counter: 'Thank you — I was expecting closer to ₹15–16 LPA. Is there flexibility?'\n3. If base is fixed, negotiate joining bonus, ESOPs, or earlier review.\n4. Never give a number first.\n5. Get everything in writing before resigning.",
    },
]
print(f"Loaded {len(SEED_EXAMPLES)} seed examples.")


Loaded 8 seed examples.


## 2.2 · Format into Alpaca instruction template

In [3]:
def format_prompt(example: dict) -> str:
    if example["input"].strip():
        return (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['output']}"
    )

# Preview one formatted example
print(format_prompt(SEED_EXAMPLES[0]))


### Instruction:
Review this resume summary and rewrite it to be stronger.

### Input:
Hardworking software developer with experience in many technologies. Looking for a challenging role where I can grow.

### Response:
Full-stack engineer with 3 years of production experience building scalable Node.js APIs and React frontends serving 50K+ users. Seeking a senior engineering role where I can drive architecture decisions and mentor junior developers.


## 2.3 · Split into train / eval and save

In [4]:
import json, random
from pathlib import Path

random.seed(42)
examples = SEED_EXAMPLES.copy()
random.shuffle(examples)

eval_ratio = 0.15
split      = max(1, int(len(examples) * eval_ratio))
train_data = examples[split:]
eval_data  = examples[:split]

Path("data").mkdir(exist_ok=True)

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for ex in data:
            record = {**ex, "text": format_prompt(ex)}
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(data)} examples → {path}")

save_jsonl(train_data, "data/train.jsonl")
save_jsonl(eval_data,  "data/eval.jsonl")


Saved 7 examples → data/train.jsonl
Saved 1 examples → data/eval.jsonl


## 2.4 · Inspect the saved data

In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "data/train.jsonl", "eval": "data/eval.jsonl"},
)
print(dataset)
print("\nSample train text:")
print("-" * 60)
print(dataset["train"][0]["text"])


Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 7
    })
    eval: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 1
    })
})

Sample train text:
------------------------------------------------------------
### Instruction:
What is the STAR method and how do I use it for interview answers?

### Response:
STAR = Situation, Task, Action, Result.
S: Set the scene briefly.
T: What was your specific responsibility?
A: What did YOU do? Use 'I', be specific.
R: Quantify the outcome (%, time saved, revenue).

Example:
S: Our deployment pipeline took 45 minutes with manual steps.
T: I was asked to reduce deployment time.
A: I automated it with GitHub Actions and added rollback triggers.
R: Deployment time dropped to 8 minutes (82% reduction).


> ✅ Data ready. Move on to `03_train.ipynb`